# ZeroBus — service principal + OAuth client secret (one-time bootstrap)

Creates (or reuses) the **workspace service principal**, mints an **OAuth client secret** when the saved one fails OIDC validation, and writes everything to the Databricks secret JSON used by **`public_example.ipynb`** (`scope` / `key` below).

- Docs: [Create a service principal and grant permissions](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#create-a-service-principal-and-grant-permissions), [OAuth M2M](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m)
- **Display name:** set optional `_SERVICE_PRINCIPAL_DISPLAY_NAME` in the first code cell, or put `ZEROBUS_SERVICE_PRINCIPAL_NAME` in the secret JSON. If both are empty, the name defaults to `zerobus-sp--<scope>--<secretKey>--<jsonField>` (same `<jsonField>` as the OAuth secret key in JSON, default `ZEROBUS_OAUTH_SECRET`). Do not use `--` inside scope, key, or field names. These apply when **creating** the SP; to rename an existing principal, use the Databricks account/workspace admin UI or the Accounts API—this notebook does not rename in place.

Then run **`public_example.ipynb`** from its config cell (or **Run All**); it merges `ZEROBUS_*` from this secret and checks that OAuth still works.

In [ ]:
import json

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, ResourceAlreadyExists, ResourceDoesNotExist

# Match public_example.ipynb (override if you use a different secret)
_SECRET_SCOPE = "lfczerobusdemo"
_SECRET_KEY = "lfczerobusdemo"

# Optional: non-empty display name for a *new* service principal (overrides secret JSON and auto name).
_SERVICE_PRINCIPAL_DISPLAY_NAME = ""

# JSON field name for OAuth client secret in the secret value; auto SP display_name uses scope/key/field if name unset.
_ZEROBUS_OAUTH_JSON_KEY = "ZEROBUS_OAUTH_SECRET"

_config_defaults = {
    "ZEROBUS_SERVICE_PRINCIPAL_NAME": "",
    "ZEROBUS_SERVICE_PRINCIPAL_ID": "",
    "ZEROBUS_APP_ID": "",
    "ZEROBUS_OAUTH_SECRET": "",
}

_w = WorkspaceClient()


def _ensure_secret_scope_and_key(w) -> None:
    try:
        w.secrets.create_scope(_SECRET_SCOPE)
        print(f"Created secret scope {_SECRET_SCOPE!r}")
    except ResourceAlreadyExists:
        pass
    try:
        w.secrets.get_secret(_SECRET_SCOPE, _SECRET_KEY)
    except ResourceDoesNotExist:
        w.secrets.put_secret(scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value="{}")
        print(f"Created empty secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r}")


def _load_saved(w) -> dict:
    try:
        raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY)
        print(f"Loaded config from secret scope={_SECRET_SCOPE!r} key={_SECRET_KEY!r}")
        return json.loads(raw)
    except Exception:
        pass
    _ensure_secret_scope_and_key(w)
    return {}


def _save_config(updates: dict, w) -> None:
    try:
        current = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
    except ResourceDoesNotExist:
        current = {}
    current.update(updates)
    w.secrets.put_secret(
        scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value=json.dumps(current, indent=2)
    )
    print(f"Saved {list(updates.keys())} to secret {_SECRET_SCOPE}/{_SECRET_KEY}")


_saved = _load_saved(_w)
_config = {
    k: (_config_defaults[k] if _config_defaults[k] else _saved.get(k, _config_defaults[k]))
    for k in _config_defaults
}
if str(_SERVICE_PRINCIPAL_DISPLAY_NAME).strip():
    _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] = str(_SERVICE_PRINCIPAL_DISPLAY_NAME).strip()
_config_original = {k: _saved.get(k, "") for k in _config_defaults}

In [ ]:
# to delete (CLI)
# databricks service-principals list --filter 'displayName co "zerobus"' --output json \
#  | jq -r '["NAME","SP_ID","APP_ID"], (.[] | [.displayName, (.id | tostring), (.applicationId | tostring)]) | @tsv' \
#  | column -t
# databricks service-principals delete <sp_id>

def _zerobus_sp_display_name(scope: str, secret_key: str, json_field: str) -> str:
    """SP display_name encodes Databricks secret scope, secret key, and JSON field for the OAuth client secret."""
    return f"zerobus-sp--{scope}--{secret_key}--{json_field}"


def _ensure_sp() -> None:
    """Ensure a ZeroBus service principal exists; updates _config in memory."""
    sp_id = _config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")

    def _create_sp():
        _name = (
            _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"]
            if _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"]
            else _zerobus_sp_display_name(_SECRET_SCOPE, _SECRET_KEY, _ZEROBUS_OAUTH_JSON_KEY)
        )
        _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] = _name
        _sp = _w.service_principals.create(display_name=_name)
        _config["ZEROBUS_SERVICE_PRINCIPAL_ID"] = str(_sp.id)
        _config["ZEROBUS_APP_ID"] = str(_sp.application_id)
        print(f"Created SP '{_name}' id={_sp.id} APP_ID={_config['ZEROBUS_APP_ID']}")

    if not sp_id:
        print("ZEROBUS_SERVICE_PRINCIPAL_ID not set — creating service principal")
        _create_sp()
    else:
        try:
            _sp = _w.service_principals.get(sp_id)
            print(f"SP exists: '{_sp.display_name}'  id={sp_id}")
            if not _config.get("ZEROBUS_APP_ID"):
                _config["ZEROBUS_APP_ID"] = str(_sp.application_id)
                print(f"Backfilled ZEROBUS_APP_ID={_config['ZEROBUS_APP_ID']}")
        except NotFound:
            print(f"SP id={sp_id} not found — creating a replacement")
            _create_sp()


_ensure_sp()
print(f"ZEROBUS_SERVICE_PRINCIPAL_ID = {_config['ZEROBUS_SERVICE_PRINCIPAL_ID']}")
print(f"ZEROBUS_APP_ID               = {_config['ZEROBUS_APP_ID']}")

In [ ]:
# (re)mint OAuth client secret if missing or OIDC check fails

def _credentials_valid(client_id: str, client_secret: str, workspace_url: str) -> bool:
    """Validate app id + secret via workspace OIDC token endpoint (no WorkspaceClient credential fallback)."""
    if not client_id or not client_secret:
        return False
    try:
        import requests as _req

        _resp = _req.post(
            f"{workspace_url.rstrip('/')}/oidc/v1/token",
            data={
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "all-apis",
            },
            timeout=10,
        )
        if not _resp.ok:
            print(f"OIDC check: {_resp.status_code} {_resp.text}")
        return _resp.ok
    except Exception as ex:
        print(f"OIDC check: {ex}")
        return False


_ws_url = _w.config.host.rstrip("/")
_ok = _credentials_valid(
    str(_config.get("ZEROBUS_APP_ID", "")).strip(),
    str(_config.get("ZEROBUS_OAUTH_SECRET", "")).strip(),
    _ws_url,
)

if _ok:
    print("OAuth client secret is valid (OIDC client_credentials).")
else:
    print("Creating new OAuth client secret for the service principal…")
    _sp_id = _config["ZEROBUS_SERVICE_PRINCIPAL_ID"]
    _secret_obj = None
    try:
        _secret_obj = _w.service_principal_secrets_proxy.create(service_principal_id=_sp_id)
    except AttributeError:
        try:
            from databricks.sdk import ServicePrincipalSecretsAPI

            _secret_obj = ServicePrincipalSecretsAPI(_w.api_client).create(service_principal_id=_sp_id)
        except Exception:
            _resp = _w.api_client.do(
                "POST",
                f"/api/2.0/accounts/servicePrincipals/{_sp_id}/credentials/secrets",
            )
            _config[_ZEROBUS_OAUTH_JSON_KEY] = _resp["secret"]

    if _secret_obj is not None:
        _config[_ZEROBUS_OAUTH_JSON_KEY] = _secret_obj.secret

    if not _credentials_valid(
        str(_config["ZEROBUS_APP_ID"]).strip(),
        str(_config[_ZEROBUS_OAUTH_JSON_KEY]).strip(),
        _ws_url,
    ):
        raise RuntimeError("New client secret failed OIDC check; verify workspace URL and SP permissions.")
    print(f"Saved new {_ZEROBUS_OAUTH_JSON_KEY} in memory; final cell persists to Databricks secret.")

In [ ]:
_changed = {k: _config[k] for k in _config_defaults if _config.get(k) != _config_original.get(k)}
if _changed:
    _save_config(_changed, _w)
else:
    print("No ZeroBus bootstrap fields changed — nothing to save.")